# RFP Proposal Generator
Extracts text from an RFP PDF and uses Claude to generate all major proposal sections with prompt caching — so the RFP is only billed once across all six section calls.

In [ ]:
%pip install -q anthropic pdfplumber pypdf PyMuPDF

In [ ]:
import os
import anthropic
import pdfplumber
from IPython.display import display, Markdown

# ── Configuration ──────────────────────────────────────────────────────────────
# Set your API key as an environment variable:  export ANTHROPIC_API_KEY=sk-ant-...
# or paste it directly below (not recommended for shared notebooks).
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

# Path to the RFP PDF
RFP_PDF_PATH = "/Users/brianpak/Desktop/Projects/Collab Architecture/1_RFP Base Template Files/2026 Master Template_Facing Pages.pdf"

# ── Firm / Team Information ─────────────────────────────────────────────────────
# Fill in your firm's details so Claude can personalize every section.
FIRM_INFO = """
FIRM NAME:          Collab Architecture
ADDRESS:            123 Main Street, Suite 400, New York, NY 10001
PHONE:              (212) 555-0100
EMAIL:              proposals@collabarchitecture.com
WEBSITE:            www.collabarchitecture.com
YEAR FOUNDED:       2010
FIRM SIZE:          45 professionals
DISCIPLINES:        Architecture, Interior Design, Urban Planning

KEY PERSONNEL:
  - Jane Smith, AIA, LEED AP — Principal-in-Charge (20 years experience)
  - Mark Johnson, RA — Project Manager (12 years experience)
  - Lisa Chen — Senior Designer (8 years experience)
  - Tom Rivera, PE — MEP Consultant (15 years experience)

NOTABLE PAST PROJECTS:
  - Hudson Yards Community Center, NYC (2023) — $12M, 45,000 sf
  - Brooklyn Public Library Renovation (2022) — $8M, 32,000 sf
  - Newark Transit Hub (2021) — $22M, 60,000 sf

BILLING RATES:
  - Principal:          $250/hr
  - Senior Architect:   $185/hr
  - Project Architect:  $145/hr
  - Designer:           $110/hr
  - CAD Technician:      $85/hr
  - Administrative:      $65/hr
"""

print("Configuration loaded. API key present:", bool(ANTHROPIC_API_KEY))

In [ ]:
def extract_rfp_text(filepath: str) -> str:
    """Extract all text from the RFP PDF using pdfplumber."""
    pages = []
    with pdfplumber.open(filepath) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text and text.strip():
                pages.append(f"[Page {i+1}]\n{text.strip()}")
    return "\n\n".join(pages)


rfp_text = extract_rfp_text(RFP_PDF_PATH)
print(f"Extracted {len(rfp_text):,} characters across {rfp_text.count('[Page')} pages.")
print("─" * 60)
print(rfp_text[:1500], "...")

In [ ]:
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """\
You are a senior proposal writer for a professional architecture and consulting firm.
Your job is to draft compelling, client-focused proposal sections based on the RFP and
firm information provided. Follow these rules:
  - Write in a professional, confident, first-person-plural voice ("our team", "we will").
  - Tailor every sentence to the specific requirements stated in the RFP.
  - Use concrete details, measurable outcomes, and relevant past project references.
  - Format output as clean Markdown suitable for copying into a proposal document.
  - Do not invent facts; if information is not available, note what would be filled in.
"""


def generate_section(section_name: str, prompt: str) -> str:
    """
    Call Claude to generate one proposal section.
    The RFP text is marked with cache_control so it is only billed once
    across all six section calls in the same session.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        f"## RFP DOCUMENT\n\n{rfp_text}\n\n"
                        f"## OUR FIRM INFORMATION\n\n{FIRM_INFO}"
                    ),
                    "cache_control": {"type": "ephemeral"},  # cached across calls
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]

    print(f"\n{'═' * 60}")
    print(f"  {section_name}")
    print(f"{'═' * 60}\n")

    result_text = ""
    with client.messages.stream(
        model="claude-opus-4-7",
        max_tokens=4096,
        system=SYSTEM_PROMPT,
        thinking={"type": "adaptive"},
        messages=messages,
    ) as stream:
        for chunk in stream.text_stream:
            print(chunk, end="", flush=True)
            result_text += chunk
        final = stream.get_final_message()

    usage = final.usage
    print(
        f"\n\n[tokens — input: {usage.input_tokens:,}  "
        f"output: {usage.output_tokens:,}  "
        f"cache_read: {usage.cache_read_input_tokens:,}  "
        f"cache_write: {usage.cache_creation_input_tokens:,}]"
    )
    return result_text


print("Client initialized. Ready to generate proposal sections.")

## Generate Proposal Sections
Run the cells below individually (or **Run All**). Each cell calls Claude once; the RFP is cached after the first call.

In [ ]:
cover_letter = generate_section(
    "1. COVER LETTER",
    """\
Write a professional cover letter (one page maximum) for our proposal responding to this RFP.
Include:
  - A compelling opening that references the specific project name and issuing agency.
  - A brief statement of our understanding of the client's core need.
  - Why our firm is uniquely qualified (reference 1–2 highly relevant past projects).
  - A commitment statement and call to action.
  - Closing with principal's name and title.
Format as a formal business letter in Markdown.
""",
)

In [ ]:
consultant_profile = generate_section(
    "2. CONSULTANT / FIRM PROFILE",
    """\
Write a Firm Profile section (approximately 400–600 words) that:
  - Introduces the firm's history, size, and core service areas.
  - Highlights our experience on projects similar in type, scale, and complexity to this RFP.
  - Describes our collaborative process and quality-control approach.
  - Notes any certifications, awards, or accreditations relevant to this project.
  - Includes a brief overview of our in-house capabilities vs. subconsultants.
Format with a bold section heading followed by structured paragraphs in Markdown.
""",
)

In [ ]:
team_qualifications = generate_section(
    "3. PROJECT TEAM QUALIFICATIONS",
    """\
Write a Project Team Qualifications section that:
  - Presents each key team member with name, title, licensure, and years of experience.
  - For each person, lists 2–3 directly relevant past projects with brief descriptions.
  - Explains each person's specific role on THIS project and their accountability.
  - Describes how the team is structured and how they will collaborate.
  - Includes a simple organizational chart description (e.g., a Markdown table or bullet hierarchy).
Format using level-3 headings for each team member, with bullet lists for their project experience.
""",
)

In [ ]:
understanding_approach = generate_section(
    "4. PROJECT UNDERSTANDING AND APPROACH",
    """\
Write a Project Understanding and Approach section (600–900 words) that:
  - Demonstrates we have read and understood the RFP in depth — reference specific
    requirements, goals, constraints, or evaluation criteria stated in the RFP.
  - Identifies the top 2–3 challenges or risks for this project and how we will address them.
  - Describes our phased work plan: key phases, milestones, and deliverables aligned
    to what the RFP requires.
  - Explains our design or consulting methodology and how it benefits this specific client.
  - Mentions community engagement, sustainability, or equity considerations if the RFP
    references them.
Format with bold phase headings and descriptive paragraphs in Markdown.
""",
)

In [ ]:
relevant_experience = generate_section(
    "5. RELEVANT PROJECT EXPERIENCE",
    """\
Write a Relevant Project Experience section that presents 3–4 past projects as case studies.
For each project include:
  - Project name, client/owner, location, year completed, and construction cost.
  - Project size (square footage, units, or other relevant metric).
  - A 3–4 sentence narrative explaining what we did and why it is relevant to THIS RFP.
  - One specific measurable outcome or award (if applicable).
  - A note on which team members worked on it.
Select and sequence projects that best match the RFP's stated scope and evaluation criteria.
Format each project as a level-3 heading with structured details beneath in Markdown.
""",
)

In [ ]:
billing_rates = generate_section(
    "6. BILLING RATES / FEE SCHEDULE",
    """\
Write a Fee Schedule / Billing Rates section that:
  - Presents our standard hourly billing rates in a clear Markdown table
    (Classification | Rate/Hour | Notes).
  - Provides a narrative paragraph explaining our fee structure, how we ensure
    budget adherence, and any not-to-exceed commitment language appropriate for this RFP.
  - If the RFP requests a lump-sum or percentage-of-construction estimate, include a
    placeholder table showing phases, estimated hours by role, and subtotals —
    with a note that final fees depend on project scope confirmation.
  - Mentions any expenses policy (reimbursable vs. included).
Format with Markdown tables and clear section labels.
""",
)

In [ ]:
# ── Compile all sections into a single Markdown document ───────────────────────
full_proposal = "\n\n---\n\n".join([
    f"# PROPOSAL\n\n{cover_letter}",
    f"## Firm Profile\n\n{consultant_profile}",
    f"## Project Team Qualifications\n\n{team_qualifications}",
    f"## Project Understanding and Approach\n\n{understanding_approach}",
    f"## Relevant Project Experience\n\n{relevant_experience}",
    f"## Fee Schedule\n\n{billing_rates}",
])

# Save to file
output_path = "/Users/brianpak/Desktop/Projects/Collab Architecture/generated_proposal.md"
with open(output_path, "w") as f:
    f.write(full_proposal)

print(f"Full proposal saved to: {output_path}")
print(f"Total characters: {len(full_proposal):,}")

# Render a preview in the notebook
display(Markdown(full_proposal[:3000] + "\n\n*... (see saved file for full proposal)*"))